# 07 - Quality & Reconciliation Report

Runs after the Silver/Gold pipeline finishes. Three things:
1. Print the recent data-quality summary (`ops.dq_summary`).
2. Print quarantine counts by rule (`ops.quarantine`).
3. Reconcile total gross sales from Bronze against Gold, and **fail the
   task** if the unexplained gap is too big — Silver's expectations already
   enforced quality; this is the human-readable check that the math adds
   up end to end.
4. If everything above passed, write a freshness marker to
   `ops.freshness_log` — this is the last task in the job, so a freshness
   row only ever appears when the full chain actually succeeded.

In [0]:
# Databricks notebook source
import json

dbutils.widgets.text("catalog", "maven_test")
catalog = dbutils.widgets.get("catalog")

In [0]:
print("=== Data Quality Summary (last 20 rule evaluations) ===")
try:
    spark.sql(f'''
        SELECT rule_id, table_name, action, records_checked, records_failed, failure_percentage, run_timestamp
        FROM {catalog}.ops.dq_summary
        ORDER BY run_timestamp DESC
        LIMIT 20
    ''').show(truncate=False)
except Exception as e:
    print(f"ops.dq_summary not queryable yet: {e}")

In [0]:
print("=== Quarantine counts by rule ===")
try:
    spark.sql(f'''
        SELECT quarantined_rule_id, quarantined_table, COUNT(*) AS row_count
        FROM {catalog}.ops.quarantine
        GROUP BY quarantined_rule_id, quarantined_table
        ORDER BY row_count DESC
    ''').show(truncate=False)
except Exception as e:
    print(f"ops.quarantine not queryable yet: {e}")

### Reconciliation: gross sales, Bronze vs Gold

A gap between Bronze and Gold totals is expected — some rows get
quarantined, some get de-duplicated. The check is whether the gap is
*fully explained* by those causes, within a 2% tolerance. Returns are
NOT part of this check on purpose: `fact_sales` is never mutated by a
return, so returns can't cause a gap here.

In [0]:
tolerance_pct = 2.0

bronze_total = spark.sql(f'''
    SELECT SUM(t.quantity * p.product_retail_price) AS total
    FROM {catalog}.bronze.transactions t
    JOIN {catalog}.bronze.bronze_mongodb_product_master p 
    ON t.product_id = p.product_id
    WHERE TRY_CAST(t.quantity AS INT) IS NOT NULL
''').collect()[0]["total"] or 0.0

gold_total = spark.sql(f"SELECT SUM(gross_sales) AS total FROM {catalog}.gold.fact_sales").collect()[0]["total"] or 0.0

quarantined_total = spark.sql(f'''
    SELECT SUM(
        CAST(get_json_object(original_row_json, '$.quantity') AS INT) * p.product_retail_price
    ) AS total
    FROM {catalog}.ops.quarantine q
    JOIN {catalog}.bronze.bronze_mongodb_product_master p
        ON CAST(get_json_object(q.original_row_json, '$.product_id') AS INT) = p.product_id
    WHERE q.quarantined_table = 'silver.transactions_clean'
      AND q.quarantined_rule_id = 'TXN_QTY_POSITIVE'
''').collect()[0]["total"] or 0.0

bronze_total = float(bronze_total)
gold_total = float(gold_total)
quarantined_total = float(quarantined_total)

raw_gap = bronze_total - gold_total
explained_gap = quarantined_total  # dedup/rescued amounts can be added here once confirmed on-workspace
unexplained_gap = raw_gap - explained_gap
unexplained_pct = (abs(unexplained_gap) / bronze_total * 100.0) if bronze_total else 0.0

reconciliation = {
    "bronze_total": bronze_total,
    "gold_total": gold_total,
    "raw_gap": raw_gap,
    "quarantined_total": quarantined_total,
    "unexplained_gap": unexplained_gap,
    "unexplained_pct": round(unexplained_pct, 4),
    "tolerance_pct": tolerance_pct,
    "status": "PASS" if unexplained_pct <= tolerance_pct else "FAIL",
}
print(json.dumps(reconciliation, indent=2))

if reconciliation["status"] == "FAIL":
    raise RuntimeError(
        f"Reconciliation FAILED: unexplained gap {reconciliation['unexplained_pct']}% "
        f"exceeds tolerance {tolerance_pct}%. See sql/validation/reconciliation.sql."
    )

### Freshness marker

Only reached if everything above passed.

In [0]:
spark.sql(f'''
    CREATE TABLE IF NOT EXISTS {catalog}.ops.freshness_log (
        run_timestamp TIMESTAMP, status STRING, job_run_id STRING
    ) USING DELTA
''')

try:
    job_run_id = dbutils.widgets.get("job_run_id")
except Exception:
    job_run_id = "manual"

spark.sql(f'''
    INSERT INTO {catalog}.ops.freshness_log
    SELECT current_timestamp(), 'SUCCESS', '{job_run_id}'
''')

print(f"Freshness marker written for catalog {catalog}.")
dbutils.notebook.exit("OK")